In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze = spark.read.table("northstar.bronze.customers_raw")

w = Window.partitionBy("customer_id").orderBy(F.col("_ingested_at").desc())

silver = (bronze
    .withColumn("rn", F.row_number().over(w))
    .filter("rn = 1").drop("rn")                              # dedup
    .withColumn("customer_city",  F.initcap(F.trim("customer_city")))   # standardize text
    .withColumn("customer_state", F.upper(F.trim("customer_state")))
    .filter(F.col("customer_id").isNotNull())                # drop bad keys
    .select("customer_id", "customer_unique_id",
            "customer_zip_code_prefix", "customer_city", "customer_state"))

silver.write.mode("overwrite").saveAsTable("northstar.silver.customers")
print("✅ silver.customers")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze = spark.read.table("northstar.bronze.orders_raw")

ts_cols = ["order_purchase_timestamp", "order_approved_at",
           "order_delivered_carrier_date", "order_delivered_customer_date",
           "order_estimated_delivery_date"]

df = bronze
for c in ts_cols:
    df = df.withColumn(c, F.to_timestamp(c))                  # string → real timestamp

w = Window.partitionBy("order_id").orderBy(F.col("_ingested_at").desc())

silver = (df
    .withColumn("rn", F.row_number().over(w))
    .filter("rn = 1").drop("rn")                              # dedup
    .filter(F.col("order_id").isNotNull() & F.col("customer_id").isNotNull())
    .select("order_id", "customer_id", "order_status", *ts_cols))

silver.write.mode("overwrite").saveAsTable("northstar.silver.orders")
print("✅ silver.orders")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze = spark.read.table("northstar.bronze.reviews_raw")
w = Window.partitionBy("review_id").orderBy(F.col("_ingested_at").desc())

silver = (bronze
    .withColumn("review_score", F.expr("try_cast(review_score as int)"))   # safe cast → NULL if bad
    .withColumn("review_creation_date",    F.to_timestamp("review_creation_date"))
    .withColumn("review_answer_timestamp", F.to_timestamp("review_answer_timestamp"))
    .withColumn("review_comment_message",  F.trim("review_comment_message"))
    .withColumn("rn", F.row_number().over(w))
    .filter("rn = 1").drop("rn")
    .filter(F.col("review_id").isNotNull())
    .filter(F.col("review_score").between(1, 5))     # keeps only valid scores
    .select("review_id", "order_id", "review_score",
            "review_comment_title", "review_comment_message",
            "review_creation_date", "review_answer_timestamp"))

silver.write.mode("overwrite").saveAsTable("northstar.silver.reviews")
print(f"✅ silver.reviews — {silver.count()} rows")

In [0]:
%sql
SELECT review_id, review_score, LEFT(review_comment_message, 60) AS msg_preview
FROM northstar.silver.reviews
WHERE review_comment_message IS NOT NULL
LIMIT 10;

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# --- products (+ English category name) ---
products = spark.read.table("northstar.bronze.products_raw")
cats     = spark.read.table("northstar.bronze.categories_raw")
wp = Window.partitionBy("product_id").orderBy(F.col("_ingested_at").desc())
(products
  .withColumn("rn", F.row_number().over(wp)).filter("rn=1").drop("rn")
  .join(cats.select("product_category_name","product_category_name_english"),
        "product_category_name","left")
  .select("product_id",
          F.coalesce("product_category_name_english","product_category_name",F.lit("unknown")).alias("category"),
          F.expr("try_cast(product_weight_g as double)").alias("weight_g"),
          F.expr("try_cast(product_photos_qty as int)").alias("photos_qty"))
  .filter(F.col("product_id").isNotNull())
  .write.mode("overwrite").saveAsTable("northstar.silver.products"))

# --- order_items (the money: price + freight) ---
(spark.read.table("northstar.bronze.order_items_raw")
  .select("order_id",
          F.expr("try_cast(order_item_id as int)").alias("order_item_id"),
          "product_id","seller_id",
          F.to_timestamp("shipping_limit_date").alias("shipping_limit_date"),
          F.expr("try_cast(price as double)").alias("price"),
          F.expr("try_cast(freight_value as double)").alias("freight_value"))
  .filter(F.col("order_id").isNotNull() & F.col("product_id").isNotNull())
  .write.mode("overwrite").saveAsTable("northstar.silver.order_items"))

# --- sellers ---
sellers = spark.read.table("northstar.bronze.sellers_raw")
ws = Window.partitionBy("seller_id").orderBy(F.col("_ingested_at").desc())
(sellers
  .withColumn("rn", F.row_number().over(ws)).filter("rn=1").drop("rn")
  .select("seller_id",
          F.initcap(F.trim("seller_city")).alias("seller_city"),
          F.upper(F.trim("seller_state")).alias("seller_state"))
  .filter(F.col("seller_id").isNotNull())
  .write.mode("overwrite").saveAsTable("northstar.silver.sellers"))
print("✅ Silver: products, order_items, sellers")